In [1]:
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [2]:
label_list = ['O', 'B-TOOLS', 'I-TOOLS', 'B-SKILL', 'I-SKILL', 'B-EXPERIENCE', 'I-EXPERIENCE', 'B-EDUCATION', 'I-EDUCATION', 'B-ROLE', 'I-ROLE']
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')
model = BertForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

c:\Users\asus\anaconda3\envs\llms\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [61]:
import os
from pathlib import Path

def read_conll_folder(folder_path):
    data = []
    for file_path in os.listdir("tagged_conll"):
        sentences = []
        labels = []
        current_sentence = []
        current_labels = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # Non-empty line
                    if len(line) > 1:
                        token, label = line.split()  # Asumsi tab/spasi pemisah
                        current_sentence.append(token)
                        current_labels.append(label)
                    else:  # Empty line = akhir dokumen/sentence
                        if current_sentence:  # Simpan kalau ada isinya
                            sentences.append(current_sentence)
                            labels.append(current_labels)
                        current_sentence = []
                        current_labels = []
            if current_sentence:  # Jangan lupa simpan dokumen terakhir
                sentences.append(current_sentence)
                labels.append(current_labels)
        data.append({"tokens": sentences, "labels": labels})
    return data

folder_path = "tagged_conll"
dataset = read_conll_folder(folder_path)

In [ ]:
flattened_data = {
    "tokens": [sentence for doc in dataset for sentence in doc["tokens"]],
    "ner_tags": [[label2id[label] for label in sentence_labels] for doc in dataset for sentence_labels in doc["labels"]]
}